# Colab P2 Extensions Notebook

Notebook one-shot cho 4 nhom phep do mo rong (P2 trong plan hoan thien):

1. **P2.1 -- Ablation sieu tham so KD**: quet `temperature`, `box_loss_weight` -> bang ablation.
2. **P2.2 -- INT8 quantization (ONNX)**: lam nhe student KD cho CPU, do mAP + latency.
3. **P2.3 -- Failure case analysis**: tim 5 anh student KD du doan kem nhat, ghep input/GT/prediction.
4. **P2.4 -- Throughput batch benchmark**: do throughput voi batch in {1,2,4,8,16} -> chon batch toi uu.

Chay tu tren xuong duoi mot lan. Cuoi notebook tu dong zip va tai ve `p2_results.zip` (chua JSON/CSV/PNG du dung cho viec tiep tuc viet bao cao).

Du kien tren Tesla T4: ~1.5-2 gio GPU. Chi phi ~4 CU (~$0.40).


## 1. CONFIG

In [ ]:
# =====================
# 1. CONFIG
# =====================
GITHUB_REPO_URL = "https://github.com/hoaianthai345/HPC_Nhom1_MLOps_ObjectDectection.git"
PROJECT_DIR = "/content/HPC_Nhom1_MLOps_ObjectDectection"

ULTRALYTICS_KD_REPO_URL = "https://github.com/ultralytics/ultralytics.git"
ULTRALYTICS_KD_DIR = "/content/ultralytics-kd"

KAGGLE_DATASET = "yusufberksardoan/traffic-detection-project"
RAW_DATA_DIR = "/content/data/raw"
SUBSET_DATA_DIR = "/content/data/demo_subset"
USE_SUBSET = True
SUBSET_SIZE = 1000

TEACHER_MODEL = "yolo26x.pt"
STUDENT_MODEL = "yolo26n.pt"
IMG_SIZE = 640
SEED = 42

# === P2.1 ABLATION GRID ===
# (label, temperature, box_w, dense_w, sparse_w, objectness_threshold)
ABLATION_GRID = [
    ("baseline",  3.0, 0.5,  0.25, 0.25, 0.3),  # cau hinh chinh thuc
    ("tau1",      1.0, 0.5,  0.25, 0.25, 0.3),  # nhiet do thap
    ("tau5",      5.0, 0.5,  0.25, 0.25, 0.3),  # nhiet do cao
    ("box_low",   3.0, 0.25, 0.25, 0.25, 0.3),  # giam trong so box
    ("box_high",  3.0, 1.0,  0.25, 0.25, 0.3),  # tang trong so box
]
ABLATION_EPOCHS = 15      # ngan hon 30 de tiet kiem
ABLATION_BATCH  = 16

# === P2.2 INT8 ===
RUN_INT8 = True

# === P2.3 FAILURE CASES ===
N_FAILURE_CASES = 5
FAILURE_IOU_THRESHOLD = 0.5

# === P2.4 THROUGHPUT BATCH ===
THROUGHPUT_BATCHES = [1, 2, 4, 8, 16]
THROUGHPUT_WARMUP = 10
THROUGHPUT_ITERS = 50

# === Optional: bundle artifacts pre-trained ===
# De khoi train lai teacher + student tu dau, neu da co bundle thi tai ve.
PRETRAINED_BUNDLE_URL = ""  # de trong = train lai tu dau

ARTIFACT_DIR = "/content/model_artifacts"
RESULTS_DIR  = "/content/p2_results"
print("Config loaded")

## 2. Runtime check + device fingerprint

In [ ]:
# =====================
# 2. RUNTIME + DEVICE
# =====================
import os, sys, json, time, shlex, shutil, platform, subprocess, statistics
from pathlib import Path

PYTHON = shlex.quote(sys.executable)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(ARTIFACT_DIR, exist_ok=True)

def run(cmd, cwd=None, check=True, capture=False):
    print(f"$ {cmd}")
    if capture:
        return subprocess.run(cmd, shell=True, cwd=cwd, check=check, capture_output=True, text=True)
    return subprocess.run(cmd, shell=True, cwd=cwd, check=check)

def pip_cmd(args, check=True):
    ip = globals().get("get_ipython", lambda: None)()
    if ip is not None:
        try: return ip.run_line_magic("pip", args)
        except Exception:
            if check: raise
            return None
    return run(f"{PYTHON} -m pip {args}", check=check)

DEVICE_INFO = {}
try:
    import torch
    DEVICE = "0" if torch.cuda.is_available() else "cpu"
    DEVICE_INFO["cuda"] = torch.cuda.is_available()
    DEVICE_INFO["torch"] = torch.__version__
    if torch.cuda.is_available():
        DEVICE_INFO["gpu"] = torch.cuda.get_device_name(0)
        DEVICE_INFO["gpu_mem_gb"] = round(torch.cuda.get_device_properties(0).total_memory/1e9, 2)
except Exception as e:
    DEVICE = "cpu"; DEVICE_INFO["torch_err"] = str(e)
DEVICE_INFO["python"] = platform.python_version()
DEVICE_INFO["platform"] = platform.platform()
try:
    DEVICE_INFO["cpu_count"] = os.cpu_count()
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemTotal"):
                DEVICE_INFO["ram_gb"] = round(int(line.split()[1])/1e6, 2); break
except Exception: pass

print("DEVICE:", DEVICE)
print(json.dumps(DEVICE_INFO, indent=2, ensure_ascii=False))

## 3. Clone repo + Ultralytics KD trainer + deps

In [ ]:
# =====================
# 3. CLONE + DEPS + KD TRAINER
# =====================
if Path(PROJECT_DIR).exists():
    run("git pull", cwd=PROJECT_DIR, check=False)
else:
    run(f"git clone {GITHUB_REPO_URL} {PROJECT_DIR}")
os.chdir(PROJECT_DIR)

pip_cmd("install --upgrade pip")
pip_cmd("install kaggle pyyaml mlflow boto3 botocore pillow opencv-python tqdm onnx onnxruntime-gpu matplotlib")
for req in ["data_pipeline/requirements.txt", "serving_pipeline/requirements.txt"]:
    if Path(req).exists():
        pip_cmd(f"install -r {req}", check=False)

if Path(ULTRALYTICS_KD_DIR).exists():
    run("git pull", cwd=ULTRALYTICS_KD_DIR, check=False)
else:
    run(f"git clone {ULTRALYTICS_KD_REPO_URL} {ULTRALYTICS_KD_DIR}")

custom_trainer = next((p for p in [Path(PROJECT_DIR)/"notebooks"/"trainer.py", Path(PROJECT_DIR)/"trainer.py"] if p.exists()), None)
assert custom_trainer is not None, "Khong tim thay trainer.py custom KD trong repo"
target_trainer = Path(ULTRALYTICS_KD_DIR)/"ultralytics"/"engine"/"trainer.py"
shutil.copy2(custom_trainer, target_trainer)
print("Copied KD trainer ->", target_trainer)

pip_cmd(f"install -e {ULTRALYTICS_KD_DIR}")
ultra_src = str(Path(ULTRALYTICS_KD_DIR).resolve())
if ultra_src not in sys.path: sys.path.insert(0, ultra_src)
import importlib; importlib.invalidate_caches()
import ultralytics
from ultralytics import YOLO
print("Ultralytics:", ultralytics.__file__)

## 4. Kaggle + dataset

In [ ]:
# =====================
# 4. DATASET
# =====================
import yaml
kaggle_json = Path.home()/".kaggle"/"kaggle.json"
if not kaggle_json.exists():
    from google.colab import files
    print("Upload kaggle.json"); up = files.upload()
    assert "kaggle.json" in up, "Can upload kaggle.json"
    kaggle_json.parent.mkdir(parents=True, exist_ok=True)
    shutil.move("kaggle.json", kaggle_json); kaggle_json.chmod(0o600)

Path(RAW_DATA_DIR).mkdir(parents=True, exist_ok=True)
r = subprocess.run(f"{PYTHON} -m data_pipeline kaggle download --dataset {KAGGLE_DATASET} --output {RAW_DATA_DIR} --organize", shell=True, cwd=PROJECT_DIR)
if r.returncode != 0:
    run(f"kaggle datasets download -d {KAGGLE_DATASET} -p {RAW_DATA_DIR} --unzip")

DATA_ROOT = RAW_DATA_DIR
if USE_SUBSET:
    rs = subprocess.run(f"{PYTHON} -m data_pipeline dataset subset --input {RAW_DATA_DIR} --output {SUBSET_DATA_DIR} --size {SUBSET_SIZE}", shell=True, cwd=PROJECT_DIR)
    if rs.returncode == 0: DATA_ROOT = SUBSET_DATA_DIR

def find_data_yaml(root):
    root = Path(root)
    c = list(root.rglob("data.yaml")) + list(root.rglob("dataset.yaml"))
    return sorted(c, key=lambda p: len(p.parts))[0] if c else None

data_yaml = find_data_yaml(DATA_ROOT) or find_data_yaml(RAW_DATA_DIR)
assert data_yaml is not None
print("DATA_YAML:", data_yaml)

## 5. Train teacher + student baseline + serving model\n\nTeacher va student baseline can co de chay ablation. Neu cung cap `PRETRAINED_BUNDLE_URL`, notebook bo qua buoc train va tai bundle ve.

In [ ]:
# =====================
# 5. TRAIN TEACHER + STUDENT BASELINE
# =====================
def newest(paths):
    paths = [p for p in paths if p.exists()]
    return sorted({p.resolve() for p in paths}, key=lambda p: p.stat().st_mtime, reverse=True)[0] if paths else None

if PRETRAINED_BUNDLE_URL:
    run(f"wget -q -O /content/bundle.zip {PRETRAINED_BUNDLE_URL}")
    run("unzip -o /content/bundle.zip -d /content/")
    teacher_best = Path("/content/model_artifacts/teacher_best.pt")
    student_best = Path("/content/model_artifacts/student_best.pt")
    assert teacher_best.exists() and student_best.exists(), "Bundle khong day du"
else:
    teacher_cmd = (f"{PYTHON} scripts/train_teacher_model.py --data {data_yaml} --model {TEACHER_MODEL} "
                   f"--epochs 30 --batch 4 --imgsz {IMG_SIZE} --device {DEVICE} "
                   f"--project runs/teacher --name teacher_yolo --no-mlflow")
    run(teacher_cmd, cwd=PROJECT_DIR)
    teacher_best = newest([*Path(PROJECT_DIR).glob('runs/**/teacher_yolo*/weights/best.pt'),
                           *Path(ULTRALYTICS_KD_DIR).glob('runs/**/teacher_yolo*/weights/best.pt')])
    student = YOLO(STUDENT_MODEL)
    sr = student.train(data=str(data_yaml), epochs=30, batch=16, imgsz=IMG_SIZE,
                       device=DEVICE, seed=SEED, project="runs/student_baseline",
                       name="student_yolo", save=True, plots=False)
    student_best = Path(sr.save_dir)/"weights"/"best.pt"

print("teacher_best:", teacher_best)
print("student_best:", student_best)
shutil.copy2(teacher_best, Path(ARTIFACT_DIR)/"teacher_best.pt")
shutil.copy2(student_best, Path(ARTIFACT_DIR)/"student_best.pt")

## P2.1 — Ablation Knowledge Distillation hyperparameters

In [ ]:
# =====================
# P2.1 ABLATION KD HYPERPARAMETERS
# =====================
import torch, csv

base_kd_cfg = Path(PROJECT_DIR)/"training_pipeline/src/config/train.yaml"
ablation_dir = Path("/content/ablation_runs"); ablation_dir.mkdir(exist_ok=True)

def read_train_metrics(pt):
    ck = torch.load(pt, map_location="cpu", weights_only=False)
    tm = ck.get("train_metrics", {}) or {}
    tt = ck.get("train_results", {}).get("time", [None])
    return {
        "precision":  tm.get("metrics/precision(B)"),
        "recall":     tm.get("metrics/recall(B)"),
        "map50":      tm.get("metrics/mAP50(B)"),
        "map5095":    tm.get("metrics/mAP50-95(B)"),
        "train_time_s": tt[-1] if tt else None,
    }

ABLATION = {}
for label, tau, box_w, dense_w, sparse_w, obj_th in ABLATION_GRID:
    print(f"\n=== Ablation: {label} (tau={tau}, box={box_w}, dense={dense_w}, sparse={sparse_w}, obj_th={obj_th}) ===")
    cfg = yaml.safe_load(base_kd_cfg.read_text())
    cfg.setdefault("training", {}).update({"epochs": ABLATION_EPOCHS, "batch": ABLATION_BATCH,
                                            "imgsz": IMG_SIZE, "device": DEVICE, "seed": SEED})
    cfg.setdefault("logging", {}).update({"project": f"runs/ablation_{label}", "name": f"kd_{label}"})
    cfg.setdefault("distillation", {}).update({
        "logit_temperature": tau,
        "dense_logit_weight": dense_w,
        "sparse_logit_weight": sparse_w,
        "box_loss_weight": box_w,
        "box_objectness_threshold": obj_th,
    })
    ab_yaml = ablation_dir/f"train_{label}.yaml"
    ab_yaml.write_text(yaml.safe_dump(cfg, sort_keys=False))

    cmd = (f"{PYTHON} training_pipeline/src/train.py {ab_yaml} "
           f"--teacher-weights {teacher_best} --student-weights {STUDENT_MODEL} "
           f"--data {data_yaml} --mlflow-tracking-uri file:/content/mlruns "
           f"--mlflow-experiment ablation --mlflow-run-name kd_{label}")
    run(cmd, cwd=PROJECT_DIR, check=False)

    cand = newest([*Path(PROJECT_DIR).glob(f"runs/**/kd_{label}*/weights/best.pt"),
                   *Path(ULTRALYTICS_KD_DIR).glob(f"runs/**/kd_{label}*/weights/best.pt")])
    if cand is None:
        print(f"[skip] {label}: khong tim thay best.pt"); continue
    meta = read_train_metrics(cand)
    meta.update({"label": label, "tau": tau, "box_w": box_w, "dense_w": dense_w,
                 "sparse_w": sparse_w, "obj_th": obj_th, "checkpoint": str(cand)})
    ABLATION[label] = meta
    print(f"  -> mAP50={meta['map50']}, mAP50-95={meta['map5095']}, P={meta['precision']}, R={meta['recall']}")

# Export
(Path(RESULTS_DIR)/"ablation_results.json").write_text(json.dumps(ABLATION, indent=2, ensure_ascii=False))
with open(Path(RESULTS_DIR)/"ablation_results.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["label","tau","box_w","dense_w","sparse_w","obj_th","precision","recall","map50","map5095","train_time_s"])
    for k, m in ABLATION.items():
        w.writerow([m["label"], m["tau"], m["box_w"], m["dense_w"], m["sparse_w"], m["obj_th"],
                    m["precision"], m["recall"], m["map50"], m["map5095"], m["train_time_s"]])

print("\n=== Bang ablation (Markdown) ===")
print("| Cau hinh | tau | box_w | P | R | mAP50 | mAP50-95 | Train (s) |")
print("|---|---|---|---:|---:|---:|---:|---:|")
for k, m in ABLATION.items():
    print(f"| {m['label']} | {m['tau']} | {m['box_w']} | {m['precision']:.3f} | {m['recall']:.3f} | {m['map50']:.3f} | {m['map5095']:.3f} | {m['train_time_s']:.0f} |")

## P2.2 — INT8 quantization (ONNX) cho CPU\n\nGiam kich thuoc va tang toc CPU bang dynamic INT8 quantization. Sau khi quant, do lai mAP va latency CPU/GPU.

In [ ]:
# =====================
# P2.2 INT8 QUANTIZATION
# =====================
import time, numpy as np
INT8_RESULTS = {}

# Lay student KD baseline (config baseline cua ablation) hoac dung serving_model.pt cu
serving_pt = Path(ARTIFACT_DIR)/"serving_model.pt"
if "baseline" in ABLATION:
    shutil.copy2(ABLATION["baseline"]["checkpoint"], serving_pt)
elif (Path(ARTIFACT_DIR)/"student_best.pt").exists():
    # fallback: dung student baseline
    shutil.copy2(Path(ARTIFACT_DIR)/"student_best.pt", serving_pt)

# Export sang ONNX (FP32)
if RUN_INT8 and serving_pt.exists():
    try:
        y = YOLO(str(serving_pt))
        y.export(format="onnx", imgsz=IMG_SIZE, dynamic=False, simplify=True)
        onnx_fp32 = Path(ARTIFACT_DIR)/"serving_model.onnx"
        # copy if not in ARTIFACT_DIR
        cand = next((p for p in [serving_pt.parent/"serving_model.onnx",
                                  Path("/content/model_artifacts/serving_model.onnx")] if p.exists()), None)
        if cand and not onnx_fp32.exists(): shutil.copy2(cand, onnx_fp32)
        print("ONNX FP32:", onnx_fp32, "size MB:", round(onnx_fp32.stat().st_size/1048576, 2))

        # Quantize INT8 dynamic
        from onnxruntime.quantization import quantize_dynamic, QuantType
        onnx_int8 = Path(ARTIFACT_DIR)/"serving_model_int8.onnx"
        quantize_dynamic(str(onnx_fp32), str(onnx_int8), weight_type=QuantType.QInt8)
        print("ONNX INT8:", onnx_int8, "size MB:", round(onnx_int8.stat().st_size/1048576, 2))

        # Val mAP (chay model.val tren onnx)
        for label, onnx_path in [("onnx_fp32", onnx_fp32), ("onnx_int8", onnx_int8)]:
            try:
                m = YOLO(str(onnx_path), task="detect")
                r = m.val(data=str(data_yaml), imgsz=IMG_SIZE, conf=0.001, iou=0.7, split="val", device=DEVICE, verbose=False)
                INT8_RESULTS[label] = {
                    "size_mb": round(onnx_path.stat().st_size/1048576, 2),
                    "map50":   round(float(r.box.map50), 4),
                    "map5095": round(float(r.box.map), 4),
                    "precision": round(float(r.box.mp), 4),
                    "recall":  round(float(r.box.mr), 4),
                }
                print(label, INT8_RESULTS[label])
            except Exception as e:
                print(f"[skip val] {label}: {e}")

        # Benchmark latency CPU + GPU (warmup + iters)
        import glob
        imgs = sorted(glob.glob(str(Path(DATA_ROOT)/"valid/images/*")))[:50] or \
               sorted(glob.glob(str(Path(DATA_ROOT)/"val/images/*")))[:50]
        for label, onnx_path in [("onnx_fp32", onnx_fp32), ("onnx_int8", onnx_int8)]:
            for dev in ["cpu", DEVICE if DEVICE != "cpu" else None]:
                if dev is None: continue
                try:
                    mm = YOLO(str(onnx_path), task="detect")
                    for i in range(10):
                        mm.predict(imgs[i % len(imgs)], imgsz=IMG_SIZE, conf=0.25, iou=0.45, device=dev, verbose=False)
                    times = []
                    for i in range(50):
                        res = mm.predict(imgs[i % len(imgs)], imgsz=IMG_SIZE, conf=0.25, iou=0.45, device=dev, verbose=False)
                        times.append(res[0].speed["inference"])
                    times = np.array(times)
                    key = f"{label}_{('cpu' if dev=='cpu' else 'gpu')}"
                    INT8_RESULTS.setdefault(key, {}).update({
                        "inf_mean_ms": round(float(times.mean()), 3),
                        "inf_p95_ms":  round(float(np.percentile(times, 95)), 3),
                        "fps":         round(1000/float(times.mean()), 1),
                    })
                    print(key, INT8_RESULTS[key])
                except Exception as e:
                    print(f"[skip bench] {label} {dev}: {e}")

    except Exception as e:
        print("INT8 pipeline failed:", e)

(Path(RESULTS_DIR)/"int8_results.json").write_text(json.dumps(INT8_RESULTS, indent=2, ensure_ascii=False))

## P2.3 — Failure case analysis\n\nXep hang anh validation theo dieu kien thieu dau: anh nao co IoU trung binh thap nhat hoac so doi tuong bo sot cao nhat. Render 5 anh worst-case (Input | Ground Truth | Prediction).

In [ ]:
# =====================
# P2.3 FAILURE CASES
# =====================
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import glob

CLASS_NAMES = ["bicycle","bus","car","motorbike","person"]
FAILURE_DIR = Path(RESULTS_DIR)/"failure_cases"; FAILURE_DIR.mkdir(exist_ok=True)

# Lay model student KD tot nhat (baseline cua ablation)
if "baseline" in ABLATION:
    kd_pt = ABLATION["baseline"]["checkpoint"]
elif (Path(ARTIFACT_DIR)/"student_best.pt").exists():
    kd_pt = str(Path(ARTIFACT_DIR)/"student_best.pt")
else:
    kd_pt = str(serving_pt)
print("Failure analysis on:", kd_pt)
m = YOLO(kd_pt)

# Lay tap val
img_glob = sorted(glob.glob(str(Path(DATA_ROOT)/"valid/images/*"))) or \
           sorted(glob.glob(str(Path(DATA_ROOT)/"val/images/*")))
label_root = Path(img_glob[0]).parent.parent/"labels" if img_glob else None
assert img_glob, "Khong co anh val"

def load_yolo_labels(label_path, img_w, img_h):
    if not Path(label_path).exists(): return []
    boxes = []
    for line in Path(label_path).read_text().splitlines():
        p = line.split()
        if len(p) < 5: continue
        cid, xc, yc, w, h = int(p[0]), *(float(x) for x in p[1:5])
        x1 = (xc - w/2)*img_w; y1 = (yc - h/2)*img_h
        x2 = (xc + w/2)*img_w; y2 = (yc + h/2)*img_h
        boxes.append((cid, x1, y1, x2, y2))
    return boxes

def iou(a, b):
    ix1 = max(a[0], b[0]); iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2]); iy2 = min(a[3], b[3])
    iw = max(0, ix2-ix1); ih = max(0, iy2-iy1)
    inter = iw*ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/ua if ua > 0 else 0

# Score per image: tinh recall + so doi tuong missed
scored = []
N_SAMPLE = min(150, len(img_glob))  # quet 150 anh dau val cho nhanh
for img_path in img_glob[:N_SAMPLE]:
    lbl_path = str(label_root/(Path(img_path).stem + ".txt"))
    img = Image.open(img_path); W, H = img.size
    gt = load_yolo_labels(lbl_path, W, H)
    if not gt: continue
    res = m.predict(img_path, imgsz=IMG_SIZE, conf=0.25, iou=0.45, device=DEVICE, verbose=False)[0]
    preds = []
    if res.boxes is not None and len(res.boxes) > 0:
        for b in res.boxes:
            cid = int(b.cls.item())
            x1,y1,x2,y2 = b.xyxy[0].tolist()
            preds.append((cid, x1, y1, x2, y2))
    # match GT vs preds at IoU >= threshold and matching class
    matched = 0
    for g in gt:
        best_iou = 0
        for p in preds:
            if p[0] != g[0]: continue
            best_iou = max(best_iou, iou((p[1],p[2],p[3],p[4]), (g[1],g[2],g[3],g[4])))
        if best_iou >= FAILURE_IOU_THRESHOLD: matched += 1
    recall_img = matched/len(gt)
    fp = len(preds) - matched
    scored.append({"img": img_path, "gt": gt, "preds": preds, "recall": recall_img,
                   "missed": len(gt) - matched, "fp": fp, "n_gt": len(gt)})

# Sort by lowest recall, then most missed
scored.sort(key=lambda s: (s["recall"], -s["missed"]))
worst = scored[:N_FAILURE_CASES]
print(f"Quet {len(scored)} anh, lay {len(worst)} anh worst.")

# Render side-by-side
def draw_boxes(img, boxes, color, names=CLASS_NAMES):
    img = img.copy()
    d = ImageDraw.Draw(img)
    for cid, x1, y1, x2, y2 in boxes:
        d.rectangle([x1,y1,x2,y2], outline=color, width=3)
        d.text((x1, max(0,y1-12)), names[cid] if cid < len(names) else str(cid), fill=color)
    return img

failure_meta = []
for i, s in enumerate(worst):
    img = Image.open(s["img"]).convert("RGB")
    img_gt = draw_boxes(img, s["gt"], "lime")
    img_pr = draw_boxes(img, s["preds"], "red")
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img);  axes[0].set_title("Input"); axes[0].axis("off")
    axes[1].imshow(img_gt); axes[1].set_title(f"GT ({s['n_gt']} objects)"); axes[1].axis("off")
    axes[2].imshow(img_pr); axes[2].set_title(f"Pred (Recall={s['recall']:.2f}, missed={s['missed']}, FP={s['fp']})"); axes[2].axis("off")
    out = FAILURE_DIR/f"failure_{i+1}.png"
    fig.tight_layout(); fig.savefig(out, dpi=120, bbox_inches="tight"); plt.close(fig)
    failure_meta.append({"index": i+1, "image": Path(s["img"]).name, "recall": s["recall"],
                          "n_gt": s["n_gt"], "missed": s["missed"], "fp": s["fp"],
                          "render": str(out.name)})
    print(f"  failure {i+1}: {Path(s['img']).name} (recall={s['recall']:.2f}, missed={s['missed']}/{s['n_gt']})")

(Path(RESULTS_DIR)/"failure_cases.json").write_text(json.dumps(failure_meta, indent=2, ensure_ascii=False))

## P2.4 — Throughput batch benchmark\n\nDo throughput cua student KD voi nhieu batch size de tim diem bao hoa cua T4.

In [ ]:
# =====================
# P2.4 THROUGHPUT BATCH
# =====================
THROUGHPUT = {}
img_pool = sorted(glob.glob(str(Path(DATA_ROOT)/"valid/images/*")))[:32] or \
           sorted(glob.glob(str(Path(DATA_ROOT)/"val/images/*")))[:32]
assert img_pool, "Khong co anh val cho throughput"

bench_model = YOLO(kd_pt if "baseline" in ABLATION else str(serving_pt))

for bs in THROUGHPUT_BATCHES:
    batch_imgs = (img_pool * ((bs // len(img_pool)) + 1))[:bs]
    # warmup
    for _ in range(THROUGHPUT_WARMUP):
        bench_model.predict(batch_imgs, imgsz=IMG_SIZE, conf=0.25, iou=0.45, device=DEVICE, verbose=False)
    t0 = time.perf_counter()
    for _ in range(THROUGHPUT_ITERS):
        bench_model.predict(batch_imgs, imgsz=IMG_SIZE, conf=0.25, iou=0.45, device=DEVICE, verbose=False)
    total = time.perf_counter() - t0
    total_imgs = THROUGHPUT_ITERS * bs
    ms_per_img = (total / total_imgs) * 1000
    THROUGHPUT[bs] = {"batch_size": bs, "total_s": round(total, 3),
                      "total_imgs": total_imgs, "ms_per_img": round(ms_per_img, 3),
                      "throughput_fps": round(total_imgs/total, 1)}
    print(f"batch={bs:2d}: {THROUGHPUT[bs]['throughput_fps']} FPS ({ms_per_img:.2f} ms/img)")

(Path(RESULTS_DIR)/"throughput.json").write_text(json.dumps(THROUGHPUT, indent=2, ensure_ascii=False))

# CSV
with open(Path(RESULTS_DIR)/"throughput.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["batch_size","total_imgs","total_s","ms_per_img","throughput_fps"])
    for bs, v in THROUGHPUT.items():
        w.writerow([v["batch_size"], v["total_imgs"], v["total_s"], v["ms_per_img"], v["throughput_fps"]])

# Plot
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6,4))
xs = list(THROUGHPUT.keys()); ys = [THROUGHPUT[b]["throughput_fps"] for b in xs]
ax.plot(xs, ys, marker="o"); ax.set_xlabel("Batch size"); ax.set_ylabel("Throughput (FPS)")
ax.set_title("Student KD throughput vs batch size (T4)")
ax.grid(True, alpha=0.3); ax.set_xticks(xs)
fig.tight_layout(); fig.savefig(Path(RESULTS_DIR)/"throughput_curve.png", dpi=130, bbox_inches="tight"); plt.close(fig)
print("Plot saved.")

## P2.5 -- So sanh literature (writing task)

Day la cong viec doc paper, khong tinh toan. Goi y lay so cu the:

- **FGD (Yang et al. 2022, CVPR)** -- Table 1 & 2: voi RetinaNet teacher (ResNet-101) -> student (ResNet-50), FGD bao cao +3.4 AP tren COCO. arXiv:2111.11837.
- **FineImitate (Wang et al. 2019, CVPR)** -- Table 2: voi Faster R-CNN ResNet-101 -> ResNet-50, fine-grained feature imitation bao cao +1.5 AP. arXiv:1906.03609.
- **Task Integration Distillation (2024)** -- Table 3 trong arXiv:2404.01699.

Chen vao bao cao Muc 4.7.3 sau khi doc tham khao tu cac paper tren va lay so cu the cho cap teacher-student gan voi cau hinh cua nhom.


## Bundle ket qua + tai ve

In [ ]:
# =====================
# BUNDLE + DOWNLOAD
# =====================
import datetime
# Manifest
manifest = {
    "generated_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "device": DEVICE_INFO,
    "config": {
        "ablation_grid": ABLATION_GRID,
        "ablation_epochs": ABLATION_EPOCHS,
        "throughput_batches": THROUGHPUT_BATCHES,
        "n_failure_cases": N_FAILURE_CASES,
        "subset_size": SUBSET_SIZE,
    },
    "ablation_count": len(ABLATION),
    "int8_keys": list(INT8_RESULTS.keys()),
    "throughput_batches_done": list(THROUGHPUT.keys()),
    "failure_cases_done": len(failure_meta) if "failure_meta" in dir() else 0,
}
(Path(RESULTS_DIR)/"manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))

# Copy ablation runs metadata
ablation_artifacts = Path(RESULTS_DIR)/"ablation_checkpoints"
ablation_artifacts.mkdir(exist_ok=True)
for label, meta in ABLATION.items():
    ck = Path(meta["checkpoint"])
    if ck.exists():
        shutil.copy2(ck, ablation_artifacts/f"kd_{label}_best.pt")

# Copy ONNX FP32 + INT8
for f in [Path(ARTIFACT_DIR)/"serving_model.onnx", Path(ARTIFACT_DIR)/"serving_model_int8.onnx"]:
    if f.exists(): shutil.copy2(f, Path(RESULTS_DIR)/f.name)

# Zip
zip_path = "/content/p2_results.zip"
if Path(zip_path).exists(): Path(zip_path).unlink()
run(f"cd /content && zip -r {zip_path} p2_results", check=False)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Tai thu cong tu /content/p2_results.zip:", e)

print("\n=== Hoan tat ===")
print("p2_results.zip chua:")
print("  - ablation_results.{json,csv} + ablation_checkpoints/*.pt")
print("  - int8_results.json + serving_model_int8.onnx")
print("  - failure_cases/*.png + failure_cases.json")
print("  - throughput.{json,csv} + throughput_curve.png")
print("  - manifest.json")

## Sau khi tai ve `p2_results.zip`

Giai nen vao `hpc_nhom1_code/reports/p2/` va dung trong bao cao:

1. **Bang ablation** (Muc 4.6 -- tao mot subsection moi "Ablation sieu tham so KD"):
   `ablation_results.csv` -> bang LaTeX 5 dong x cot mAP50/mAP50-95/P/R/Train_time.

2. **Bang INT8** (Muc 4.5 -- bo sung dong "Student KD ONNX INT8" vao Bang latency):
   `int8_results.json` -> map size MB + mAP + latency CPU/GPU.

3. **Failure case figure** (Muc 4.7 hoac 4.8 -- tao mot subsection "Truong hop that bai"):
   5 anh `failure_cases/failure_*.png`.

4. **Throughput curve** (Muc 4.5 -- bo sung phan "Throughput batch"):
   `throughput_curve.png` + `throughput.csv`.

5. **Literature numbers** (P2.5 -- task viet): doc FGD/FineImitate va dien so cu the vao Muc 4.7.3.
